# Volatility Forecasting Full Run
This notebook is a standalone end-to-end workflow that:
- installs required packages (`arch`, `statsforecast`, `flaml`, and optional `autogluon.timeseries`)
- loads stock data and builds `Volatility` as target if needed
- runs `BrainAutoMLForecast` using `brain_automl`
- runs volatility baselines per stock: EWMA(0.94), GARCH, EGARCH, and GJR-GARCH
- saves all outputs to `examples/volatility_output` (metrics, forecasts, plots, summaries)

Run cells top-to-bottom once.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import warnings

warnings.filterwarnings("ignore")

CWD = Path.cwd()
# Handle both possible launch roots: project root or examples folder
if (CWD / "src").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "src").exists():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

candidate_data = [
    PROJECT_ROOT / "examples" / "dataset.csv",
    PROJECT_ROOT / "dataset.csv",
    CWD / "dataset.csv",
]
DATA_PATH = next((p for p in candidate_data if p.exists()), candidate_data[0])

OUTPUT_DIR = PROJECT_ROOT / "examples" / "volatility_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CWD:", CWD)
print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Output path:", OUTPUT_DIR)

# Ensure src is importable
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

def pip_install(packages):
    if not packages:
        return
    cmd = [sys.executable, "-m", "pip", "install", *packages]
    print("Installing:", " ".join(packages))
    subprocess.check_call(cmd)

# Required for volatility models/backends
required_pkgs = ["arch", "statsforecast", "flaml", "seaborn", "plotly"]
try:
    pip_install(required_pkgs)
    print("Required packages installed.")
except Exception as e:
    print("Package installation issue:", e)
    print("Continuing with available packages.")

# Optional heavy package for richer backend coverage in some setups
INSTALL_AUTOGLUON = False  # set True if you want to attempt AutoGluon install
if INSTALL_AUTOGLUON:
    try:
        pip_install(["autogluon.timeseries"])
        print("AutoGluon installed.")
    except Exception as e:
        print("AutoGluon install failed:", e)
        print("Proceeding without AutoGluon.")

CWD: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples
Project root: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai
Data path: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/dataset.csv
Output path: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output
Installing: arch statsforecast flaml seaborn plotly
Required packages installed.


In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

required_cols = {"Date", "Stock", "Close"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date", "Stock", "Close"]).copy()
df = df.sort_values(["Stock", "Date"]).reset_index(drop=True)

# Build volatility if missing (squared log returns)
if "Volatility" not in df.columns:
    df["_logret"] = np.log(df["Close"] / df.groupby("Stock")["Close"].shift(1))
    df["Volatility"] = (df["_logret"] ** 2).fillna(0.0)
    df = df.drop(columns=["_logret"])
else:
    if df["Volatility"].isna().any():
        df["_logret"] = np.log(df["Close"] / df.groupby("Stock")["Close"].shift(1))
        df["Volatility"] = df["Volatility"].fillna((df["_logret"] ** 2))
        df["Volatility"] = df["Volatility"].fillna(0.0)
        df = df.drop(columns=["_logret"])


print("Prepared data shape:", df.shape)
print("Stocks:", df["Stock"].nunique())
display(df.head())

df.to_csv(OUTPUT_DIR / "dataset_with_volatility.csv", index=False)
print("Saved:", OUTPUT_DIR / "dataset_with_volatility.csv")

Prepared data shape: (18381, 12)
Stocks: 5


,Date,Close,High,Low,Open,Volume,Return,MA_5,MA_10,MA_20,Volatility,Stock
0,2010-02-01,8649.149414,8712.148681,8495.151205,8597.350212,0,-0.000266,8550.110352,8779.747852,8929.211182,7.068376e-08,BANKNIFTY
1,2010-02-02,8468.051758,8718.548840,8453.801924,8666.599250,0,-0.021161,8504.421094,8717.238574,8897.001562,4.477675e-04,BANKNIFTY
2,2010-02-03,8631.749023,8662.849247,8489.700874,8489.700874,0,0.019147,8564.830273,8666.834082,8868.986816,3.665956e-04,BANKNIFTY
3,2010-02-04,8471.000977,8623.149596,8455.401549,8591.699767,0,-0.018798,8574.280078,8601.309863,8831.392236,3.533825e-04,BANKNIFTY
4,2010-02-05,8223.154297,8319.053571,8150.355339,8319.053571,0,-0.029695,8488.621094,8531.705664,8782.907764,8.817818e-04,BANKNIFTY


Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/dataset_with_volatility.csv


## Run BrainAutoML + Volatility Models
The next cells execute full modeling and artifact generation into `examples/volatility_output`.

In [3]:
from brain_automl.model_zoo.time_series_ai import BrainAutoMLForecast

HORIZONS = [7, 15, 30]
MAX_H = max(HORIZONS)

# Configure a broad but practical run
model = BrainAutoMLForecast(
    horizon=MAX_H,
    timestamp_column="Date",
    target_column="Volatility",
    item_id_column="Stock",
    include_backends=True,
    include_comprehensive=True,
    include_advanced_dl=False,
    output_dir=str(OUTPUT_DIR),
)

# One train pass per stock using max horizon holdout; evaluate all horizons
model.fit_all_stocks(df, horizons=HORIZONS)

multi_lb = model.multi_stock_leaderboard()
multi_lb.to_csv(OUTPUT_DIR / "brain_automl_volatility_leaderboard.csv", index=False)
print("Saved:", OUTPUT_DIR / "brain_automl_volatility_leaderboard.csv")
print("Leaderboard rows:", len(multi_lb))
display(multi_lb.head(20))

[22:37:44] INFO     Brain-AI run started
[22:37:44] INFO     Log file : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/logs/brain_automl_20260406_223744_932751.log
[22:37:44] INFO     Output dir: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output
[22:37:44] INFO     ============================================================
[22:37:44] INFO     fit_all_stocks() starting
[22:37:44] INFO       Data shape  : (18381, 12)
[22:37:44] INFO       Horizons    : [7, 15, 30]  (max=30)
[22:37:44] INFO       Target      : Volatility
[22:37:44] INFO       Item-ID col : 'Stock'
[22:37:44] INFO     ============================================================
[22:37:44] INFO     Stocks to process: ['BANKNIFTY', 'HDFCBANK', 'NIFTY50', 'RELIANCE', 'TCS'] (5 total)
[22:37:44] INFO     
──────────────────────────────────────────────────
[22:37:44] INFO     [1/5] Stock: BANKNIFTY
[22:

Backends:   0%|          | 0/3 [00:00<?, ?backend/s]

[22:37:45] INFO     [START] autogluon_timeseries


[22:38:23] INFO     [ OK ] autogluon_timeseries — 38.4s | RMSE=0.0001  MAE=0.0001
[22:38:23] INFO     [START] statsforecast
[22:38:46] INFO     [ OK ] statsforecast — 22.9s | RMSE=0.0001  MAE=0.0001
[22:38:46] INFO     [START] flaml
[22:39:33] INFO     [ OK ] flaml — 47.2s | RMSE=0.0001  MAE=0.0001
[22:39:33] INFO     Best backend: autogluon_timeseries | RMSE=0.0001  MAE=0.0001
[22:39:33] INFO     forecast_last_horizon complete
[22:39:33] INFO     Saved per-stock predictions to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/predictions/BANKNIFTY_predictions_h30.csv
[22:39:33] [comprehensive] Starting: 1 stock(s), 1 column(s), 3 decompositions, 5 models  →  up to 15 combinations
[22:39:33] [comprehensive] ── Stock: BANKNIFTY  (train=3647, test=30 rows)
[22:39:33] [comprehensive] combo 1/15 | BANKNIFTY/Volatility | decomp=original model=ARIMA
[22:39:34] [comprehensive]    ✓ RMSE=0.0001  MAE=0.0001  MAPE=222806.3203
[22:39:3

Backends:   0%|          | 0/3 [00:00<?, ?backend/s]

...

[23:35:06] INFO     [ OK ] autogluon_timeseries — 26.8s | RMSE=0.0002  MAE=0.0002
[23:35:06] INFO     [START] statsforecast
[23:35:35] INFO     [ OK ] statsforecast — 29.0s | RMSE=0.0002  MAE=0.0001
[23:35:35] INFO     [START] flaml
[23:36:07] INFO     [ OK ] flaml — 32.0s | RMSE=0.0002  MAE=0.0002
[23:36:07] INFO     Best backend: statsforecast | RMSE=0.0002  MAE=0.0001
[23:36:07] INFO     forecast_last_horizon complete
[23:36:07] INFO     Saved per-stock predictions to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/predictions/RELIANCE_predictions_h30.csv
[23:36:07] [comprehensive] Starting: 1 stock(s), 1 column(s), 3 decompositions, 5 models  →  up to 15 combinations
[23:36:07] [comprehensive] ── Stock: RELIANCE  (train=3651, test=30 rows)
[23:36:07] [comprehensive] combo 1/15 | RELIANCE/Volatility | decomp=original model=ARIMA
[23:36:08] [comprehensive]    ✓ RMSE=0.0002  MAE=0.0001  MAPE=4451.5912
[23:36:08] [comprehe

Backends:   0%|          | 0/3 [00:00<?, ?backend/s]

[23:46:48] INFO     [START] autogluon_timeseries


[23:47:12] INFO     [ OK ] autogluon_timeseries — 23.2s | RMSE=0.0004  MAE=0.0002
[23:47:12] INFO     [START] statsforecast
[23:47:18] INFO     [ OK ] statsforecast — 6.8s | RMSE=0.0003  MAE=0.0002
[23:47:18] INFO     [START] flaml
[23:48:02] INFO     [ OK ] flaml — 43.6s | RMSE=0.0003  MAE=0.0002
[23:48:02] INFO     Best backend: statsforecast | RMSE=0.0003  MAE=0.0002
[23:48:02] INFO     forecast_last_horizon complete
[23:48:02] INFO     Saved per-stock predictions to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/predictions/TCS_predictions_h30.csv
[23:48:02] [comprehensive] Starting: 1 stock(s), 1 column(s), 3 decompositions, 5 models  →  up to 15 combinations
[23:48:02] [comprehensive] ── Stock: TCS  (train=3651, test=30 rows)
[23:48:02] [comprehensive] combo 1/15 | TCS/Volatility | decomp=original model=ARIMA
[23:48:02] [comprehensive]    ✓ RMSE=0.0004  MAE=0.0002  MAPE=1948.5729
[23:48:02] [comprehensive] combo 2/1

,stock,horizon,model,library,mae,mse,rmse,mape,smape,wape,mase,r2,column
0,BANKNIFTY,7,decomposition+model/XGBoost,classical,0.000069,NaN,0.000097,9.775872e+04,NaN,NaN,NaN,NaN,Volatility
1,BANKNIFTY,7,decomposition+model/LSTM,classical,0.000071,NaN,0.000100,9.457607e+04,NaN,NaN,NaN,NaN,Volatility
2,BANKNIFTY,7,decomposition+model/RandomForest,classical,0.000070,NaN,0.000103,8.827417e+04,NaN,NaN,NaN,NaN,Volatility
3,BANKNIFTY,7,original/XGBoost,classical,0.000087,NaN,0.000105,1.620531e+05,NaN,NaN,NaN,NaN,Volatility
4,NIFTY50,7,original/XGBoost,classical,0.000070,NaN,0.000111,1.048311e+07,NaN,NaN,NaN,NaN,Volatility
5,NIFTY50,7,original/RandomForest,classical,0.000076,NaN,0.000112,1.362447e+07,NaN,NaN,NaN,NaN,Volatility
6,NIFTY50,7,original/ExpSmoothing,classical,0.000074,NaN,0.000112,1.311795e+07,NaN,NaN,NaN,NaN,Volatility
7,NIFTY50,7,original/ARIMA,classical,0.000076,NaN,0.000113,1.410533e+07,NaN,NaN,NaN,NaN,Volatility
8,NIFTY50,7,original/LSTM,classical,0.000068,NaN,0.000113,1.045032e+07,NaN,NaN,NaN,NaN,Volatility
9,BANKNIFTY,7,original/ExpSmoothing,classical,0.000101,NaN,0.000115,2.089606e+05,NaN,NaN,NaN,NaN,Volatility


In [4]:
# Best model summaries
best_per_stock = model.best_model_per_stock()
overall_best = model.overall_best_models()

best_per_stock.to_csv(OUTPUT_DIR / "best_model_per_stock_rmse.csv", index=False)
overall_best.to_csv(OUTPUT_DIR / "overall_best_models_rmse.csv", index=False)

print("Saved:", OUTPUT_DIR / "best_model_per_stock_rmse.csv")
print("Saved:", OUTPUT_DIR / "overall_best_models_rmse.csv")
display(best_per_stock.head(20))
display(overall_best.head(20))

Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/best_model_per_stock_rmse.csv
Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/overall_best_models_rmse.csv


,stock,horizon,model,library,mae,mse,rmse,mape,smape,wape,mase,r2,column
0,BANKNIFTY,7,decomposition+model/XGBoost,classical,0.000069,1.323460e-08,0.000097,9.775872e+04,102.789333,80.051318,0.297010,0.407382,Volatility
1,BANKNIFTY,15,decomposition+model/XGBoost,classical,0.000069,9.554397e-09,0.000097,9.775872e+04,113.636682,108.466433,0.262799,0.263593,Volatility
2,BANKNIFTY,30,decomposition+model/XGBoost,classical,0.000069,9.964374e-09,0.000097,9.775872e+04,118.614252,100.945358,0.236006,-0.011117,Volatility
3,HDFCBANK,7,WeightedEnsemble,autogluon_timeseries,0.000097,1.863601e-08,0.000137,2.629944e+06,97.519582,62.270316,0.331825,0.380446,Volatility
4,HDFCBANK,15,decomposition+model/XGBoost,classical,0.000103,2.072093e-08,0.000141,3.170437e+05,116.337430,104.136686,0.401921,0.050984,Volatility
5,HDFCBANK,30,AutoARIMA,statsforecast,0.000112,1.879401e-08,0.000137,9.314938e+05,119.055112,115.636517,0.384697,-0.190378,Volatility
6,NIFTY50,7,original/XGBoost,classical,0.000070,3.340457e-08,0.000111,1.048311e+07,103.372609,80.035763,0.789550,0.012115,Volatility
7,NIFTY50,15,original/XGBoost,classical,0.000070,1.662980e-08,0.000111,1.048311e+07,119.809807,84.405292,0.491257,0.159010,Volatility
8,NIFTY50,30,arima,flaml,0.000065,1.157540e-08,0.000108,2.223929e+06,135.519937,93.469617,0.432021,0.083796,Volatility
9,RELIANCE,7,original/XGBoost,classical,0.000155,1.074908e-07,0.000218,5.114803e+03,75.033337,65.794041,0.517481,0.164967,Volatility


,model,library,horizon,mean_rmse,std_rmse,n_stocks
0,original/XGBoost,classical,7,0.000189,0.000102,5
1,decomposition+model/LSTM,classical,7,0.000191,0.000113,5
2,original/RandomForest,classical,7,0.000191,0.000098,5
3,decomposition+model/XGBoost,classical,7,0.000194,0.000119,5
4,decomposition+model/RandomForest,classical,7,0.000195,0.000116,5
5,original/XGBoost,classical,15,0.000189,0.000102,5
6,decomposition+model/LSTM,classical,15,0.000191,0.000113,5
7,original/RandomForest,classical,15,0.000191,0.000098,5
8,decomposition+model/XGBoost,classical,15,0.000194,0.000119,5
9,decomposition+model/RandomForest,classical,15,0.000195,0.000116,5


In [5]:
# Run volatility baselines per stock (EWMA, GARCH-family via arch)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from arch import arch_model
    arch_available = True
except Exception as e:
    arch_available = False
    print("arch unavailable:", e)

garch_dir = OUTPUT_DIR / "garch_egarch"
garch_dir.mkdir(parents=True, exist_ok=True)

def metric_row(stock, model_name, horizon, y_true, y_pred):
    return {
        "stock": stock,
        "model_col": model_name,
        "horizon": int(horizon),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mse": float(mean_squared_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0),
        "r2": float(r2_score(y_true, y_pred)),
    }

garch_rows = []
for stock in sorted(df["Stock"].unique()):
    sdf = df[df["Stock"] == stock].sort_values("Date").copy()
    sdf["ret"] = np.log(sdf["Close"]).diff().fillna(0.0) * 100.0

    # Align evaluation to last max horizon points
    eval_idx = sdf.index[-MAX_H:] if len(sdf) >= MAX_H else sdf.index
    actual_vol_all = sdf.loc[eval_idx, "Volatility"].to_numpy(dtype=float)
    if len(actual_vol_all) < 2:
        print(f"Skipping {stock}: not enough observations for baseline evaluation")
        continue

    # EWMA (RiskMetrics-style) baseline with lambda=0.94 for daily data
    lam = 0.94
    ret = sdf["ret"].to_numpy(dtype=float) / 100.0
    ewma_var = np.zeros_like(ret, dtype=float)
    ewma_var[0] = np.var(ret) if len(ret) > 1 else 0.0
    for i in range(1, len(ret)):
        ewma_var[i] = lam * ewma_var[i - 1] + (1.0 - lam) * (ret[i - 1] ** 2)
    ewma_vol = np.sqrt(np.maximum(ewma_var, 0.0))
    ewma_pred_all = ewma_vol[-len(eval_idx):]

    # Evaluate EWMA for each horizon separately
    for h in HORIZONS:
        if len(actual_vol_all) >= h:
            actual_vol_h = actual_vol_all[-h:]
            ewma_pred_h = ewma_pred_all[-h:]
            garch_rows.append(metric_row(stock, "ewma_0p94", h, actual_vol_h, ewma_pred_h))

    # Save full predictions for the largest horizon
    pred_df_ewma = pd.DataFrame({
        "Date": sdf.loc[eval_idx, "Date"].values,
        "actual_volatility": actual_vol_all,
        "predicted_volatility": ewma_pred_all
    })
    pred_df_ewma.to_csv(garch_dir / f"{stock}_ewma_0p94_predictions.csv", index=False)
    print(f"Saved EWMA predictions for {stock}")

    if arch_available:
        arch_specs = [
            {"name": "garch", "vol": "Garch", "kwargs": {"p": 1, "q": 1}},
            {"name": "egarch", "vol": "EGARCH", "kwargs": {"p": 1, "q": 1}},
            {"name": "gjr_garch", "vol": "GARCH", "kwargs": {"p": 1, "o": 1, "q": 1}},
        ]
        for spec in arch_specs:
            try:
                am = arch_model(
                    sdf["ret"],
                    mean="Zero",
                    vol=spec["vol"],
                    dist="normal",
                    **spec["kwargs"],
                )
                res = am.fit(disp="off")
                cond_vol = (res.conditional_volatility / 100.0).reindex(sdf.index).to_numpy(dtype=float)
                pred_vol_all = cond_vol[-len(eval_idx):]

                # Evaluate each GARCH variant for each horizon separately
                for h in HORIZONS:
                    if len(actual_vol_all) >= h:
                        actual_vol_h = actual_vol_all[-h:]
                        pred_vol_h = pred_vol_all[-h:]
                        garch_rows.append(metric_row(stock, spec["name"], h, actual_vol_h, pred_vol_h))

                # Save full predictions for the largest horizon
                pred_df = pd.DataFrame({
                    "Date": sdf.loc[eval_idx, "Date"].values,
                    "actual_volatility": actual_vol_all,
                    "predicted_volatility": pred_vol_all
                })
                pred_df.to_csv(garch_dir / f"{stock}_{spec['name']}_predictions.csv", index=False)
                print(f"Saved {spec['name']} predictions for {stock}")
            except Exception as e:
                print(f"{spec['name']} failed for {stock}: {e}")
    else:
        print("Skipping arch-based models because arch is not installed.")

garch_metrics = pd.DataFrame(garch_rows)
if not garch_metrics.empty:
    garch_metrics.to_csv(OUTPUT_DIR / "garch_egarch_metrics.csv", index=False)
    print("Saved:", OUTPUT_DIR / "garch_egarch_metrics.csv")
display(garch_metrics.head(20))


Saved EWMA predictions for BANKNIFTY
Saved garch predictions for BANKNIFTY
Saved egarch predictions for BANKNIFTY
Saved gjr_garch predictions for BANKNIFTY
Saved EWMA predictions for HDFCBANK
Saved garch predictions for HDFCBANK
Saved egarch predictions for HDFCBANK
Saved gjr_garch predictions for HDFCBANK
Saved EWMA predictions for NIFTY50
Saved garch predictions for NIFTY50
Saved egarch predictions for NIFTY50
Saved gjr_garch predictions for NIFTY50
Saved EWMA predictions for RELIANCE
Saved garch predictions for RELIANCE
Saved egarch predictions for RELIANCE
Saved gjr_garch predictions for RELIANCE
Saved EWMA predictions for TCS
Saved garch predictions for TCS
Saved egarch predictions for TCS
Saved gjr_garch predictions for TCS
Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/garch_egarch_metrics.csv


,stock,model_col,horizon,mae,mse,rmse,mape,r2
0,BANKNIFTY,ewma_0p94,7,0.009245,0.000086,0.009252,2.028301e+05,-11054.368630
1,BANKNIFTY,ewma_0p94,15,0.008964,0.000081,0.008975,5.784715e+06,-12097.378925
2,BANKNIFTY,ewma_0p94,30,0.009651,0.000094,0.009687,3.368612e+06,-9568.304976
3,BANKNIFTY,garch,7,0.010238,0.000105,0.010247,2.246907e+05,-13560.238507
4,BANKNIFTY,garch,15,0.009760,0.000096,0.009779,6.177651e+06,-14363.231388
5,BANKNIFTY,garch,30,0.010460,0.000110,0.010501,3.610010e+06,-11245.826056
6,BANKNIFTY,egarch,7,0.010547,0.000112,0.010563,2.328942e+05,-14411.200488
7,BANKNIFTY,egarch,15,0.009914,0.000099,0.009946,6.245370e+06,-14857.003280
8,BANKNIFTY,egarch,30,0.010744,0.000117,0.010804,3.672298e+06,-11904.182156
9,BANKNIFTY,gjr_garch,7,0.011062,0.000123,0.011074,2.418259e+05,-15838.357851


In [6]:
# Merge BrainAutoML and volatility-baseline metrics into one comparison table
brain_metrics = multi_lb.copy()
brain_metrics = brain_metrics.rename(columns={"model": "model_col"})
brain_metrics["source"] = "brain_automl"

if not garch_metrics.empty:
    garch_cmp = garch_metrics.copy()
    garch_cmp["source"] = "vol_baseline"
    common_cols = sorted(set(brain_metrics.columns).intersection(set(garch_cmp.columns)))
    combined = pd.concat([brain_metrics[common_cols], garch_cmp[common_cols]], ignore_index=True)
else:
    combined = brain_metrics.copy()

combined.to_csv(OUTPUT_DIR / "combined_volatility_metrics.csv", index=False)
print("Saved:", OUTPUT_DIR / "combined_volatility_metrics.csv")
display(combined.head(20))

pub_summary = (
    combined.groupby(["source", "model_col", "horizon"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_mae=("mae", "mean"),
        mean_mape=("mape", "mean"),
        mean_r2=("r2", "mean"),
        n_stocks=("stock", "nunique")
    )
    .sort_values(["horizon", "mean_rmse"])
    .reset_index(drop=True)
)

pub_summary.to_csv(OUTPUT_DIR / "publication_summary_by_horizon.csv", index=False)
print("Saved:", OUTPUT_DIR / "publication_summary_by_horizon.csv")
display(pub_summary.head(30))

Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/combined_volatility_metrics.csv


,horizon,mae,mape,model_col,mse,r2,rmse,source,stock
0,7,0.000069,9.775872e+04,decomposition+model/XGBoost,NaN,NaN,0.000097,brain_automl,BANKNIFTY
1,7,0.000071,9.457607e+04,decomposition+model/LSTM,NaN,NaN,0.000100,brain_automl,BANKNIFTY
2,7,0.000070,8.827417e+04,decomposition+model/RandomForest,NaN,NaN,0.000103,brain_automl,BANKNIFTY
3,7,0.000087,1.620531e+05,original/XGBoost,NaN,NaN,0.000105,brain_automl,BANKNIFTY
4,7,0.000070,1.048311e+07,original/XGBoost,NaN,NaN,0.000111,brain_automl,NIFTY50
5,7,0.000076,1.362447e+07,original/RandomForest,NaN,NaN,0.000112,brain_automl,NIFTY50
6,7,0.000074,1.311795e+07,original/ExpSmoothing,NaN,NaN,0.000112,brain_automl,NIFTY50
7,7,0.000076,1.410533e+07,original/ARIMA,NaN,NaN,0.000113,brain_automl,NIFTY50
8,7,0.000068,1.045032e+07,original/LSTM,NaN,NaN,0.000113,brain_automl,NIFTY50
9,7,0.000101,2.089606e+05,original/ExpSmoothing,NaN,NaN,0.000115,brain_automl,BANKNIFTY


Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/publication_summary_by_horizon.csv


,source,model_col,horizon,mean_rmse,mean_mae,mean_mape,mean_r2,n_stocks
0,brain_automl,original/XGBoost,7,0.000189,0.000130,2.273282e+06,NaN,5
1,brain_automl,decomposition+model/LSTM,7,0.000191,0.000116,1.603804e+06,NaN,5
2,brain_automl,original/RandomForest,7,0.000191,0.000138,2.890246e+06,NaN,5
3,brain_automl,decomposition+model/XGBoost,7,0.000194,0.000115,1.191433e+06,NaN,5
4,brain_automl,decomposition+model/RandomForest,7,0.000195,0.000114,1.197863e+06,NaN,5
5,brain_automl,original/ExpSmoothing,7,0.000196,0.000138,2.842919e+06,NaN,5
6,brain_automl,original/ARIMA,7,0.000202,0.000145,3.070872e+06,NaN,5
7,brain_automl,decomposition+model/ARIMA,7,0.000210,0.000115,8.111130e+05,NaN,5
8,brain_automl,original/LSTM,7,0.000217,0.000161,2.343015e+06,NaN,5
9,brain_automl,decomposition+model/ExpSmoothing,7,0.000218,0.000119,2.382389e+05,NaN,5


In [7]:
# Plots and saved artifacts
import plotly.express as px

plot_dir = OUTPUT_DIR / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

if not pub_summary.empty:
    fig_rmse = px.bar(
        pub_summary,
        x="model_col",
        y="mean_rmse",
        color="source",
        facet_col="horizon",
        title="Volatility Forecast RMSE Comparison by Horizon"
    )
    fig_rmse.write_html(plot_dir / "rmse_comparison.html")
    print("Saved:", plot_dir / "rmse_comparison.html")

    fig_rank = px.line(
        pub_summary.sort_values(["horizon", "mean_rmse"]),
        x="horizon",
        y="mean_rmse",
        color="model_col",
        line_dash="source",
        markers=True,
        title="RMSE Trend Across Horizons"
    )
    fig_rank.write_html(plot_dir / "rmse_trend.html")
    print("Saved:", plot_dir / "rmse_trend.html")

# Final run manifest
manifest = pd.DataFrame([
    {"artifact": "brain_automl_volatility_leaderboard.csv"},
    {"artifact": "best_model_per_stock_rmse.csv"},
    {"artifact": "overall_best_models_rmse.csv"},
    {"artifact": "garch_egarch_metrics.csv"},
    {"artifact": "combined_volatility_metrics.csv"},
    {"artifact": "publication_summary_by_horizon.csv"},
    {"artifact": "plots/rmse_comparison.html"},
    {"artifact": "plots/rmse_trend.html"},
])
manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)
print("Saved:", OUTPUT_DIR / "run_manifest.csv")
display(manifest)

Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/plots/rmse_comparison.html
Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/plots/rmse_trend.html
Saved: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/volatility_output/run_manifest.csv


,artifact
0,brain_automl_volatility_leaderboard.csv
1,best_model_per_stock_rmse.csv
2,overall_best_models_rmse.csv
3,garch_egarch_metrics.csv
4,combined_volatility_metrics.csv
5,publication_summary_by_horizon.csv
6,plots/rmse_comparison.html
7,plots/rmse_trend.html
